In [ ]:
import re
import time
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import local

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

thread_local = local()


def get_driver():
    if not hasattr(thread_local, "driver"):
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")

        # BLOCK IMAGE -> tăng tốc mạnh
        prefs = {
            "profile.managed_default_content_settings.images": 2,
            "profile.managed_default_content_settings.stylesheets": 2,
            "profile.managed_default_content_settings.fonts": 2
        }
        options.add_experimental_option("prefs", prefs)

        driver = webdriver.Chrome(options=options)
        driver.set_page_load_timeout(15)

        thread_local.driver = driver

    return thread_local.driver


# SAVE CSV
def save_to_csv(data, filename='../data/raw/thuviennhadat_raw.csv'):
    if not data:
        print("Không có dữ liệu")
        return

    fields = [
        "tieu_de", "gia", "dia_chi",
        "dien_tich_dat", "phong_ngu", "phong_tam",
        "so_tang", "phap_ly", "ngay_dang"
    ]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in data:
            writer.writerow({k: row.get(k, "") for k in fields})

    print(f"Đã lưu {len(data)} dòng")


# EXTRACT SỐ TẦNG
def extract_so_tang(text):
    text = text.lower().replace("tầng", "tang")
    pattern = re.findall(r'(\d{1,2})\s*tang|\btang\s*(\d{1,2})', text)

    nums = [int(m[0] or m[1]) for m in pattern if (m[0] or m[1])]
    return max(nums) if nums else None

# SCRAPE DETAIL (FAST)
def scrape_detail(url, retry=2):
    driver = get_driver()

    for _ in range(retry):
        try:
            driver.get(url)

            # load nhanh hơn WebDriverWait
            time.sleep(0.8)

            soup = BeautifulSoup(driver.page_source, "html.parser")

            title_tag = soup.find("p", class_="text-truncate-2")
            if not title_tag:
                return None

            data = {}
            tieu_de = title_tag.get_text(strip=True)

            data["tieu_de"] = tieu_de
            data["dia_chi"] = tieu_de.split("tại")[-1].strip() if "tại" in tieu_de.lower() else tieu_de

            items = soup.select("ul.detailroom li")

            for item in items:
                title = item.get("title", "")
                value_tag = item.find("span")
                value = value_tag.get_text(strip=True) if value_tag else None

                if not value:
                    continue

                if "Mức giá" in title:
                    data["gia"] = value
                elif "Diện tích" in title:
                    data["dien_tich_dat"] = value
                elif "Phòng ngủ" in title:
                    data["phong_ngu"] = value
                elif "Phòng tắm" in title:
                    data["phong_tam"] = value
                elif "Pháp lý" in title:
                    data["phap_ly"] = value

            #NGÀY ĐĂNG (FIX CHUẨN)
            ngay_dang = None

            # Cách 1: giống code cũ (ổn định)
            ngay = soup.find("div", string=lambda x: x and "Ngày đăng" in x)
            if ngay:
                val = ngay.find_next("div", class_="text-primary")
                if val:
                    ngay_dang = val.get_text(strip=True)

            # Cách 2: fallback (an toàn hơn nếu DOM đổi)
            if not ngay_dang:
                candidates = soup.select("div.text-primary")
                for c in candidates:
                    txt = c.get_text(strip=True)
                    if re.search(r"\d{1,2}/\d{1,2}/\d{2,4}", txt):
                        ngay_dang = txt
                        break

            data["ngay_dang"] = ngay_dang


            # SỐ TẦNG 
            text = soup.get_text(" ", strip=True)
            data["so_tang"] = extract_so_tang(text)

            
            return data

        except Exception:
            time.sleep(0.5)

    return None


# SCRAPE LIST PAGE (GIỮ 1 DRIVER)
def scrape_list_page(url):
    driver = get_driver()

    try:
        driver.get(url)
        time.sleep(0.7)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        links = set()
        for a in soup.select("div.navigateTo a[href*='pst']"):
            href = a.get("href")
            if href and href.startswith("/"):
                links.add("https://thuviennhadat.vn" + href)

        return list(links)

    except Exception:
        return []


def crawl_full(max_pages=10, target=500, start_page=1):
    base = "https://thuviennhadat.vn/ban-nha-tai-dinh-cu-thanh-pho-ha-noi?cIds=6,5,4,3"

    results = []

    with ThreadPoolExecutor(max_workers=8) as executor:

        # 👇 sửa chỗ này
        for page in range(start_page, start_page + max_pages):
            url = base if page == 1 else f"{base}&trang={page}"

            links = scrape_list_page(url)
            if not links:
                continue

            print(f"Trang {page}: {len(links)} link")

            futures = [executor.submit(scrape_detail, link) for link in links]

            for f in as_completed(futures):
                data = f.result()
                if data:
                    results.append(data)

                    print(f"✔ {len(results)}", data.get("gia"))

                    if len(results) >= target:
                        return results

    return results


if __name__ == "__main__":
    data = crawl_full(start_page=1, max_pages=600, target=5500)

    print("Tổng:", len(data))
    save_to_csv(data)